# Inspect BirdPark `alignment.nwb`

Loads the alignment sidecar for the downloaded BirdPark example dataset and inspects
how each media stream's timing is stored.

For a constant-rate audio file the `audio_mic-1` ImageSeries should use **compact**
`rate` + `starting_time` + `num_samples` metadata (with `timestamps = None`), rather than
a dense per-sample `timestamps` array. Dense storage is what made the sidecar balloon to
tens of MB at high sample rates.

In [ ]:
from pathlib import Path

from ethograph.datasets import dataset_dir

align_path = dataset_dir("birdpark") / ".ethograph" / "alignment.nwb"
print("path  :", align_path)
print("exists:", align_path.exists())
print("size  :", f"{align_path.stat().st_size / 1024:.1f} KB" if align_path.exists() else "-")

## Raw pynwb view — per-stream timing metadata

`timestamps_len = None` + a set `rate`/`num_samples` means compact storage (good).
A large `timestamps_len` means dense per-sample storage (the old, bloated path).

In [ ]:
from pynwb import NWBHDF5IO

with NWBHDF5IO(str(align_path), "r") as io:
    nwb = io.read()
    print("session_description:", nwb.session_description)
    print("trials:")
    print(nwb.trials.to_dataframe())
    print("\nacquisition streams:")
    for name, acq in nwb.acquisition.items():
        ts = getattr(acq, "timestamps", None)
        print(
            f"  {name:14s} | rate={getattr(acq, 'rate', None)} "
            f"| starting_time={getattr(acq, 'starting_time', None)} "
            f"| num_samples={getattr(acq, 'num_samples', None)} "
            f"| timestamps_len={len(ts) if ts is not None else None} "
            f"| external_file={list(acq.external_file)}"
        )

## Via `NWBAlignment` — the interface the GUI actually uses

In [ ]:
from ethograph.io.nwb_alignment import NWBAlignment

align = NWBAlignment(align_path)

print("cameras       :", align.cameras)
print("mics          :", align.mics)
print("video rate    :", align.get_stream_rate("video", align.cameras[0] if align.cameras else None))
print("audio rate    :", align.get_stream_rate("audio", align.mics[0] if align.mics else None))
print("trials_df:")
print(align.trials_df)

# Reconstructed file time spans (uses num_samples in compact rate mode)
for mic in align.mics:
    print(f"\naudio '{mic}' file_time_spans:", align.file_time_spans("audio", mic))
for cam in align.cameras:
    print(f"video '{cam}' file_time_spans:", align.file_time_spans("video", cam))

align.close()

## Sanity check: how big would the *dense* equivalent have been?

Compares the actual sidecar size against the size a dense per-sample `timestamps`
array would have required (float64 = 8 bytes/sample), summed across streams.

In [ ]:
with NWBHDF5IO(str(align_path), "r") as io:
    nwb = io.read()
    dense_bytes = 0
    for name, acq in nwb.acquisition.items():
        n = getattr(acq, "num_samples", None)
        ts = getattr(acq, "timestamps", None)
        n = int(n) if n is not None else (len(ts) if ts is not None else 0)
        dense_bytes += n * 8

actual_kb = align_path.stat().st_size / 1024
print(f"actual sidecar size          : {actual_kb:8.1f} KB")
print(f"dense-timestamps equivalent  : {dense_bytes / 1024 / 1024:8.1f} MB  (per-sample float64)")